In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

**<font size="6" color="red">ch5. LSTM/GRU</font>**
- 5만개 영화 감상평(독립변수) -> 부정/긍정(타겟변수)

## 1. 패키지

In [ ]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from time import time #70.1.1부터 현재까지의 밀리세컨

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score

## 2. 하이퍼 파리미터 설정(이 파라미터를 바꾸면 정확도나 학습 속도에 차이남)

In [ ]:
MY_WORDS  = 10000 # imdb 데이터의 단어수
MY_LENGTH = 80    # 영화평 단어수 80개만 독립변수
MY_EMBED  = 32    # Embedding layer의 결과 차원
MY_HIDDEN = 64    # LSTM의 units 차원

MY_EPOCH  = 10    # 학습 수(fit)
MY_BATCH  = 200   # batch_size(fit시 매번 데이터를 가져오는 데이터)

## 3. 데이터 불러오기

In [ ]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MY_WORDS)

In [ ]:
print('학습셋 입력변수 모양 :', x_train.shape)
print('학습셋 타겟변수 모양 :', y_train.shape)
print('학습셋 샘플 :', type(x_train[2]), x_train[100], y_train[2])
print('테스트셋 변수들 모양 :', x_test.shape, y_test.shape)

In [ ]:
# 긍정/부정 갯수
print('학습셋의 긍정 갯수 :', y_train.sum())
print('테스트셋의 긍정 갯수 :', y_test.sum())

## 4. 문자단어 -> 정수

In [ ]:
word_to_id = imdb.get_word_index() # {'word':id}
print(word_to_id['movie'])
print(word_to_id['film'])
print(word_to_id['sonja'])
print(word_to_id['a'])
print(word_to_id['the'])
# 정수 -> 문자 단어
id_to_word = {} #{1:'the', 3: 'a', 16816:'sonja'}
for word, value in word_to_id.items():
    id_to_word[value] = word
print(id_to_word[1])
print(id_to_word[3])

In [ ]:
msg = "What a wonderful movie"
msg = msg.lower().split()
# 1:리뷰시작을 알리는 숫자, 2:문자가짤려서잘못읽어옴, 3:padding처리
data = [1] + [word_to_id.get(m, -1)+3 for m in msg]
print('원 후기 내용 :', msg)
print('encoded된 data :', data)
print('추정된 data :', [id_to_word.get(d-3, '???') for d in data])
print('추정된 data :', ' '.join([id_to_word.get(d-3, '???') for d in data]))

## 5. 숫자영화평 -> 자연어 영화평 return 함수

In [ ]:
def decoding(review_num):
    decoded = [id_to_word.get(num-3, '???') for num in review_num]
    return ' '.join(decoded)

In [ ]:
print(decoding(x_train[1]), y_train[1])

## 6. 영화평 (입력변수)의 길이

In [ ]:
def show_length(x_train):
    print('첫 20개 영화평 길이')
    print([len(x_data) for x_data in x_train[:20]])

In [ ]:
# pad_sequence 전
show_length(x_train)

In [ ]:
print('제일 긴 영화평 단어 길이 :', 
     max(len(x_data) for x_data in x_train))
print('제일 짧은 영화평 단어 길이 :', 
     min(len(x_data) for x_data in x_train))
print('영화평 단어 길이 중위수 :',
     np.median([len(x_data) for x_data in x_train]))

## 7. 모든 영화평 길이를 동일하게(MY_LENGTH=80)

In [ ]:
X_train = pad_sequences(x_train,
                       padding='post',
                       truncating='post', # 뒷부분을 짜르고 앞부분을 남김
                       maxlen=MY_LENGTH)
X_test = pad_sequences(x_test,
                      padding='post',
                      truncating='post',
                      maxlen=MY_LENGTH)
show_length(X_train), show_length(X_test)

# 8. 최종 데이터 shape 확인

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
print(x_train[0])

In [ ]:
X_train[0]

# 9. 모델 생성

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=MY_WORDS, # 10000
                   output_dim=MY_EMBED, # 32
                   input_length=MY_LENGTH)) # 80
# RNN : 입력 단어의 길이 수가 너무 길면 파리미터 업데이트 안 됨
# 개선모델1. LSTM 개선모델2. GRU
model.add(LSTM(units=MY_HIDDEN,
              input_shape=(MY_LENGTH, MY_EMBED)
         ))
model.add(Dense(units=1, activation='sigmoid'))
model.summary()

# 10. 학습환경 설정 및 학습하기


In [ ]:
model.compile(loss='binary_crossentropy', # 이진분류시 손실함수
             optimizer='adam',
             metrics=['acc'])
begin = time() # 70.1.1부터 현재까지의 밀리세컨
hist = model.fit(X_train, y_train,
                epochs=MY_EPOCH,
                batch_size=MY_BATCH,
                validation_split=0.2,
                verbose=1)
end = time() # 70.1.1부터 이 시점까지의 밀리세컨
print('총 학습시간 :', (end-begin))

In [ ]:
hist.history.keys()

In [ ]:
# 5. 모델 학습과정 시각화
import matplotlib.pyplot as plt
fig, loss_ax = plt.subplots(figsize=(12,6))
loss_ax.plot(hist.history['loss'], 'y', label='train loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val loss')
acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['acc'], 'g', label='train accuracy')
acc_ax.plot(hist.history['val_acc'], 'b', label='val accuracy')
loss_ax.set_xlabel('epochs')
loss_ax.set_ylabel('loss')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right')
acc_ax.legend(loc='upper left')
plt.show()

In [ ]:
# pad_sequences 'post' : 75.81%
# pad_sequences 'pre' : 80.19%
loss, acc = model.evaluate(X_test, y_test)
print('정확도 :', acc)

# 11. 모델 평가

In [ ]:
# pad_sequences 'post' : 75.81%
# pad_sequences 'pre' : 80.19%
# MY_LENGTH=178: 83.89%
loss, acc = model.evaluate(X_test, y_test)
print('정확도 :', acc)

In [ ]:
# 혼동행렬, recall, precision을 위한 yhat
yhat = (model.predict(X_test, 
                      verbose=0) > 0.5).astype(np.int16).reshape(-1)
yhat

In [ ]:
# 혼동행렬
confusion_matrix(y_test, yhat)

In [ ]:
# recall(실제 True인 것 중 True로 예측한 비율) 10238/(2262+10238)
recall_score(y_test, yhat)

In [ ]:
# precision(True로 예측한 것 중 실제값이 True인 비율) 10238/(2778+10238)
precision_score(y_test, yhat)

# 12. 모델 사용하기

In [ ]:
review = """The movie was really exciting, 
and I watched it throughout the movie hoping it wouldn't end without 
watching the clock. I strongly recommend it. 
It was even more interesting thinking that it could happen in real life. 
The main character was handsome and acted well, so my eyes were happy, 
and the story was fun @_@ㅠ.ㅠ""".lower()
import re
review = re.sub('[^a-zA-Z\'\s]', '', review)
review = review.split() # 단어 list
review = [1] + [word_to_id.get(word, -1)+3 for word in review]
print(review, len(review))

In [ ]:
input_data = pad_sequences([review],
                      padding='pre',
                      truncating='pre',
                      maxlen=MY_LENGTH)
input_data

In [ ]:
result = (model.predict(input_data)>0.5).astype('int8').reshape(-1)
result